In [1]:
from nnterp import load_model
import torch as th
th.set_grad_enabled(False)
import gc
gemma_2 = load_model("google/gemma-2-2b", device_map="cuda:0")
gemma_2_it = load_model("google/gemma-2-2b-it", device_map="cuda:0")

In [2]:
from tiny_dashboard import OfflineFeatureCentricDashboard
from huggingface_hub import hf_hub_download
repo_id = "Butanium/max-activating-examples-gemma-2-2b-l13-mu4.1e-02-lr1e-04"

In [3]:
import pandas as pd
df_path = hf_hub_download(repo_id=repo_id, filename="feature_df.csv", repo_type="dataset")
df = pd.read_csv(df_path, index_col=0)
available_features = df[
    (df["tag"].isin(["IT only", "Base only"])) & (df["dead"] == False)
]
available_features_idx = available_features.index.tolist()
available_features.sort_values(by='base uselessness score', ascending=False).head(10)

feature_df.csv:   0%|          | 0.00/64.5M [00:00<?, ?B/s]

,tag,dead,dec_norm_diff,base uselessness score,avg_activation,lmsys_ctrl_%,lmsys_bos_%,lmsys_user_%,lmsys_assistant_%,lmsys_dead,...,max_act_lmsys_train,max_act_lmsys_val,max_act_fw_train,max_act_fw_val,beta_activation_no_bias_base,beta_activation_no_bias_chat,beta_activation_no_bias_ratio,twin_div_fw_val,twin_div_lmsys_val,twin_div_val
latent,,,,,,,,,,,,,,,,,,,,,
67022,IT only,False,0.020710,11.086456,2.032327,0.002175,0.0,0.969295,0.030705,False,...,22.834978,18.943615,16.291918,15.660289,11.974142,19.976364,0.599415,NaN,NaN,NaN
34224,IT only,False,0.014979,11.044545,1.877579,0.001004,0.0,0.977492,0.022508,False,...,20.272924,21.889917,17.814245,17.542540,12.083431,19.754181,0.611690,NaN,NaN,NaN
35421,IT only,False,0.010672,10.948557,2.279533,0.000394,0.0,0.942319,0.057681,False,...,22.870186,20.517622,23.047644,20.181463,7.632565,14.460939,0.527806,NaN,NaN,NaN
42348,IT only,False,0.018160,10.629179,1.891290,0.018809,0.0,0.904069,0.095931,False,...,17.256788,14.763035,18.225847,16.927805,14.208382,23.590141,0.602302,NaN,NaN,NaN
27301,IT only,False,0.014230,10.544223,2.275819,0.002095,0.0,0.886023,0.113977,False,...,24.925879,24.214184,16.628695,17.046997,16.108339,23.402555,0.688315,NaN,NaN,NaN
1045,IT only,False,0.022567,10.495647,2.844598,0.003067,0.0,0.889883,0.110117,False,...,37.715816,37.331280,28.960247,13.291065,5.471940,9.793116,0.558754,NaN,NaN,NaN
55798,IT only,False,0.020818,10.478554,3.179345,0.112859,0.0,0.972354,0.027646,False,...,30.806606,28.869600,24.008022,21.435440,11.669093,20.288797,0.575150,NaN,NaN,NaN
14939,IT only,False,0.033090,10.414942,3.852006,0.193194,0.0,0.893820,0.106180,False,...,57.504433,48.537067,33.027840,31.870785,6.895824,13.633035,0.505817,NaN,NaN,NaN
56854,IT only,False,0.027670,10.381703,3.097182,0.032212,0.0,0.716818,0.283182,False,...,24.577955,23.516012,25.818933,21.382391,12.270437,20.597794,0.595716,NaN,NaN,NaN


In [4]:
# Most frequent features on lmsys
available_features.sort_values(by='lmsys_freq', ascending=False).head(5)

,tag,dead,dec_norm_diff,base uselessness score,avg_activation,lmsys_ctrl_%,lmsys_bos_%,lmsys_user_%,lmsys_assistant_%,lmsys_dead,...,max_act_lmsys_train,max_act_lmsys_val,max_act_fw_train,max_act_fw_val,beta_activation_no_bias_base,beta_activation_no_bias_chat,beta_activation_no_bias_ratio,twin_div_fw_val,twin_div_lmsys_val,twin_div_val
latent,,,,,,,,,,,,,,,,,,,,,
39490,Base only,False,0.992711,NaN,1.469457,0.000376,0.0,0.018953,0.981047,False,...,13.477418,11.727644,7.749556,5.402079,NaN,NaN,NaN,NaN,NaN,NaN
53366,Base only,False,0.991999,NaN,2.526684,0.004762,0.0,0.095365,0.904635,False,...,24.485151,25.683910,18.209553,17.526171,NaN,NaN,NaN,NaN,NaN,NaN
56811,Base only,False,0.993355,NaN,0.842609,0.035992,0.0,0.217127,0.782873,False,...,10.071166,8.166938,9.995775,9.174471,NaN,NaN,NaN,NaN,NaN,NaN
50034,IT only,False,0.015542,0.633294,2.099340,0.010342,0.0,0.087706,0.912294,False,...,18.123438,16.687414,11.484550,11.031973,12.865059,18.134457,0.709426,NaN,NaN,NaN
21528,Base only,False,0.989044,NaN,1.552147,0.000012,0.0,0.082527,0.917473,False,...,17.761023,18.822012,8.092329,7.138829,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
feature_idx = 62595 # @param {type:"raw"}
df[df.index == feature_idx]

,tag,dead,dec_norm_diff,base uselessness score,avg_activation,lmsys_ctrl_%,lmsys_bos_%,lmsys_user_%,lmsys_assistant_%,lmsys_dead,...,max_act_lmsys_train,max_act_lmsys_val,max_act_fw_train,max_act_fw_val,beta_activation_no_bias_base,beta_activation_no_bias_chat,beta_activation_no_bias_ratio,twin_div_fw_val,twin_div_lmsys_val,twin_div_val
feature,,,,,,,,,,,,,,,,,,,,,
62595,IT only,False,0.023991,0.942073,3.770604,0.010866,0.0,0.216299,0.783701,False,...,46.268353,35.939774,40.285034,38.867458,6.777925,9.323519,0.726971,0.424779,0.481735,0.470054


In [5]:
# @markdown # Download max activating examples database
examples_source = "chat data" # @param ["chat data", "web data", "both chat and web"]
# @markdown Use 20 for faster download
num_max_activating_examples = 20 # @param [20, 100] {type:"raw"}
match examples_source:
    case "chat data":
        name = "chat"
    case "web data":
        name = "base"
    case "both chat and web":
        name = "chat_base"
num = "_20" if num_max_activating_examples == 20 else ""
db_path = hf_hub_download(repo_id=repo_id, filename=f"{name}_examples{num}.db", repo_type="dataset")
gc.collect()

0

In [6]:
dashboard = OfflineFeatureCentricDashboard.from_db(db_path, gemma_2_it.tokenizer, column_name="entries")
dashboard.display()

CrossCoder inference:

In [9]:
from dictionary_learning import CrossCoder
from nnsight import LanguageModel
import torch as th
import gc
crosscoder = CrossCoder.from_pretrained("Butanium/gemma-2-2b-crosscoder-l13-mu4.1e-02-lr1e-04", from_hub=True, device="cuda")
gc.collect()
prompt = "quick fox brown"

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

In [4]:
from dictionary_learning import CrossCoder
from nnsight import LanguageModel
import torch as th
import gc
crosscoder = CrossCoder.from_pretrained("Butanium/gemma-2-2b-crosscoder-l13-mu4.1e-02-lr1e-04", from_hub=True, device="cuda")
gc.collect()
prompt = "quick fox brown"

Online Dashboard

In [2]:
from tiny_dashboard.dashboard_implementations import CrosscoderOnlineFeatureDashboard

In [5]:
crosscoder_dashboard = CrosscoderOnlineFeatureDashboard(
    gemma_2, gemma_2_it, crosscoder, collect_layer=13,
)
crosscoder_dashboard.display()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

FileNotFoundError: [Errno 2] No such file or directory: 'results/features/10/1741177295.html'

FileNotFoundError: [Errno 2] No such file or directory: 'results/features/10/1741177308.html'